## LinkedIn Profile Scraper in Python using LLM

In [ ]:
!pip install python-dotenv selenium beautifulsoup4

In [71]:
import warnings
warnings.filterwarnings("ignore")

In [72]:
import os
from dotenv import load_dotenv
load_dotenv()

False

In [47]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By

In [48]:
driver = webdriver.Chrome()

In [49]:
driver.get('https://www.linkedin.com/login')

In [50]:
driver.title

'LinkedIn Login, Sign in | LinkedIn'

In [51]:
email = driver.find_element(By.ID, 'username')
email.send_keys(os.getenv('EMAIL'))

In [53]:
password = driver.find_element(By.ID, 'password')
password.send_keys(os.getenv('PASSWORD'))

In [54]:
password.submit()

In [55]:
## MAKE SURE TO USE ONLY THIS URL TO AVOID BEING STUCK IN ERRORS

url = "https://www.linkedin.com/in/{your_linkedin_profile_url}"

driver.get(url)

In [78]:
page_source = driver.page_source

In [81]:
soup = BeautifulSoup(page_source, 'lxml')

In [82]:
# soup.get_text()

"Neel Contractor | LinkedInNeel Contractor | LinkedIn0 notificationsSkip to main contentHomeMy NetworkJobsMessaging23NotificationsMeFor BusinessTry Premium for ₹0Neel ContractorSolana Developer (Anchor) | Shipping production dApps end-to-end (smart contracts + full-stack) | Seeking high-impact roles at top tech & crypto firmsEnhance profileAdd sectionOpen to Neel ContractorAdd verification badgeSolana Developer (Anchor) | Shipping production dApps end-to-end (smart contracts + full-stack) | Seeking high-impact roles at top tech & crypto firmsAmity UniversityAhmedabad, Gujarat, India·Contact infoAmity UniversityOpen toAdd sectionEnhance profileOpen to workBlockchain Developer and Blockchain Intern rolesShow detailsSuggested for youPrivate to youNeel, increase your visibility and credibilityStand out with a custom call-to-action, multiple cover photos, and more. Subscribers see up to 13x more profile views.Millions of members use PremiumTry Premium for ₹01-month free trial. Cancel whenev

In [75]:
profile = soup.find('main', {'class': 'IDbhLWzXdzKoCEksNaayQTAEeGRjvNDI'})

In [76]:
# profile

In [ ]:
# sections = profile.find_all('section', {'class': 'artdeco-card'})

In [ ]:
# len(sections)

In [ ]:
# sections

In [ ]:
# sections_text = [section.get_text() for section in sections]

In [83]:
import re

# remove multiple new lines and tabs
def clean_text(text):
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'\t+', '\t', text)
    text = re.sub(r'\t\s+', ' ', text)
    text = re.sub(r'\n\s+', '\n', text)
    return text

In [ ]:
sections_text = [clean_text(section) for section in sections_text]

In [68]:
def remove_duplicates(text):
    lines = text.split('\n')
    new_lines = []
    for line in lines:
        if line[:len(line)//2] == line[len(line)//2:]:
            new_lines.append(line[:len(line)//2])
        else:
            new_lines.append(line)

    return '\n'.join(new_lines)

In [ ]:
sections_text = [remove_duplicates(section) for section in sections_text]

In [ ]:
print(sections_text[1])

In [ ]:
sections_text

## Profile Data Parsing using LLM

In [85]:
from langchain_ollama import ChatOllama

from langchain_core.prompts import (SystemMessagePromptTemplate, 
                                    HumanMessagePromptTemplate,
                                    ChatPromptTemplate)



from langchain_core.output_parsers import StrOutputParser

base_url = "http://localhost:11434"
model = 'qwen3:0.6b'
# model = "qwen2.5:7b"

llm = ChatOllama(base_url=base_url, model=model)


system = SystemMessagePromptTemplate.from_template("""You are helpful AI assistant who answer LinkedIn profile parsing related 
                                                    user question based on the provided profile text data.""")

def ask_llm(prompt):
    prompt = HumanMessagePromptTemplate.from_template(prompt)

    messages = [system, prompt]
    template = ChatPromptTemplate(messages)

    qna_chain = template | llm | StrOutputParser()

    return qna_chain.invoke({})

In [86]:
llm

ChatOllama(model='qwen3:0.6b', base_url='http://localhost:11434')

In [87]:
ask_llm("hello")

'Hello! How can I assist you today? 😊'

In [ ]:
template = """
Extract and return the requested information from the LinkedIn profile data in a concise, point-by-point format (up to 5 points). Avoid preambles or any additional context.

### LinkedIn Profile Data:
{}

### Information to Extract:
Extract '{}' in bullet points, limiting the output to 5 points. Provide only the necessary details.
Remember, It is LinkedIn profile data.

### Extracted Data:"""

context = sections_text[0]
k = "Name and Headline"

prompt = template.format(context, k).replace('{', '{{').replace('}', '}}')
response = ask_llm(prompt)

In [ ]:
# print(prompt)

In [ ]:
print(response)

In [ ]:
section_keys = ['Name and Headline']
for section in sections_text[1:]:
    # print(section.strip().split('\n'))
    section_keys.append(section.strip().split('\n')[0])

section_keys

In [ ]:
# sections_text
responses = {}

for k,context in zip(section_keys, sections_text):
    prompt = template.format(context, k).replace('{', '{{').replace('}', '}}')
    response = ask_llm(prompt=prompt)
    responses[k] = response

In [ ]:
print(responses)

In [ ]:
import json

with open('linkedin_profile_data.json', 'w') as f:
    json.dump(responses, f, indent=4)

## Second Level of LLM Call

In [ ]:
template = """You are provided with LinkedIn profile data in JSON format.
            Parse the data according to the specified schema, correct any spelling errors,
            and condense the information if possible.

### LinkedIn Profile JSON Data:
{context}

### Schema You need to follow:
You need to extract
Name:
Headline:
About:
Experience:
Education:
Skills:
Projects:
Summary:

Do not return preambles or any other information.
### Parsed Data:"""

prompt = template.format(context=responses).replace("{", "{{").replace("}", "}}")
response = ask_llm(prompt=prompt)
# response

In [ ]:
print(response)